This notebook is to practice training a Graph Neural Network on the Prometheus database.

My goal is to graph all species, reactions and mechanism connections, and train on ChemKED, paraventure we could predict combustion kinetics outcomes.

**Step 1: Construct the training corpus.**
From Prometheus, extract for each mechanism:
- The species graph: nodes = species, features = [ΔfH°(298K), S°(298K), Cp(300K), ..., Cp(2000K), molecular weight, number of atoms]
- The reaction graph: edges connecting reactant and product species, features = [A, n, Ea, k(500K), k(1000K), k(1500K), k(2000K)]
- The mechanism identity: which research group, which year, which fuel

**Step 2: Train a graph neural network.**
Architecture: Message Passing Neural Network (MPNN) or Graph Attention Network (GAT) operating on the reaction graph.

Pre-training task (self-supervised): **masked parameter prediction**. Randomly mask 15% of rate constants or NASA polynomial coefficients. Train the GNN to predict the masked values from the surrounding network context. This forces the model to learn the relationships between rate constants, thermochemistry, and network position.

This is the chemical kinetics analog of BERT's masked language modeling — the model learns that a rate constant for an H-abstraction reaction by OH "should" have a certain value given the thermochemistry of the reactant and the rates of similar reactions in the network.

**Step 3: Fine-tune for downstream tasks.**
- **Anomaly detection:** Which rate constants in a new mechanism are outliers relative to the learned representation? (Flag potential errors)
- **Missing parameter prediction:** For a new mechanism with incomplete rate expressions, predict the missing values from context
- **Performance prediction:** Given a mechanism's graph representation, predict its validation error against ChemKED without running Cantera (orders of magnitude faster screening)

**Step 4: Evaluate transfer learning.**
Train on mechanisms for fuel A (e.g., H₂/CH₄). Fine-tune on a small number of mechanisms for fuel B (e.g., NH₃). Test whether the foundation model improves over training from scratch — demonstrating that the learned representations capture transferable combustion chemistry knowledge.